# 01 - Best Practice Analyzer scan, with last-scan vs this-scan

Scans one semantic model with the Best Practice Analyzer from Semantic Link Labs,
saves the findings, and compares them with the previous scan so you see what is
**NEW** and what is **FIXED**. The scan is not the value. The comparison is.

**Before you run it**
1. Attach a lakehouse to this notebook (the scan files are written under `Files/bpa`).
2. Set `MODEL` and `MODEL_WORKSPACE` in the first code cell.
3. You need Read and Write permission on the model (`run_model_bpa` requires it).

**Known limits (seen with semantic-link-labs 0.18.1)**
- Semantic Link Labs rule *Direct Lake using a view* crashed on a Direct Lake model
  (`numpy vectorize on size 0 inputs`). The rule's code runs numpy `vectorize` on a query result
  with no check for an empty result, which is the likely trigger (not re-tested).
  `SKIP_DIRECT_LAKE_VIEW_RULE = True` switches that one rule off, so it is **not evaluated**.
  Set it to `False` to try again on a newer version of the library.
- The library's own save-to-table option failed in a Python (non-Spark) notebook
  (`schema_mode 'merge' is not supported in pyarrow engine`), so this notebook saves
  timestamped CSV files instead.
- The notebook reports only the rules that fired, not how many rules were checked. Do not
  quote a rule total from this output.


In [ ]:
# ---- settings: change these ----
MODEL = "Your_Semantic_Model"
MODEL_WORKSPACE = "Your Workspace Name"      # name or ID of the workspace that holds the model
SKIP_DIRECT_LAKE_VIEW_RULE = True            # see "Known limits" above
SCAN_DIR = "/lakehouse/default/Files/bpa"    # needs a default lakehouse attached


In [ ]:
# Unpinned on purpose for the first run. After you test it, pin the version you tested.
%pip install semantic-link-labs -q


In [ ]:
# ---- tested helper code (src/bpa_diff.py) ----
import pandas as pd

# A finding is "the same finding" when these columns match. Severity is left out
# on purpose: a rule changing severity is not a new finding.
BPA_KEYS = ["Category", "Rule Name", "Object Type", "Object Name"]


def diff_scans(previous, current, keys=BPA_KEYS):
    """Compare two BPA scans (pandas DataFrames).

    Returns a dict of three DataFrames:
      new       - in `current` but not in `previous`
      fixed     - in `previous` but not in `current`
      unchanged - in both

    `previous` may be None or empty (first scan): everything is then NEW.
    """
    cur = current.drop_duplicates(subset=keys).reset_index(drop=True)
    if previous is None or len(previous) == 0:
        return {"new": cur, "fixed": cur.iloc[0:0], "unchanged": cur.iloc[0:0]}

    prev = previous.drop_duplicates(subset=keys).reset_index(drop=True)
    merged = cur.merge(prev[keys], on=keys, how="outer", indicator=True)
    new = merged[merged["_merge"] == "left_only"].drop(columns="_merge").reset_index(drop=True)
    unchanged = merged[merged["_merge"] == "both"].drop(columns="_merge").reset_index(drop=True)
    fixed = (
        prev.merge(cur[keys], on=keys, how="left", indicator=True)
        .query("_merge == 'left_only'")
        .drop(columns="_merge")
        .reset_index(drop=True)
    )
    return {"new": new, "fixed": fixed, "unchanged": unchanged}


def summarize(df):
    """Count findings by Category and Severity (the 'findings by category' view)."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["Category", "Severity", "Findings"])
    return (
        df.groupby(["Category", "Severity"]).size().reset_index(name="Findings")
        .sort_values(["Category", "Severity"]).reset_index(drop=True)
    )


In [ ]:
import os, glob
from datetime import datetime, timezone
import sempy_labs as labs

if SKIP_DIRECT_LAKE_VIEW_RULE:
    from sempy_labs.tom._model import TOMWrapper
    TOMWrapper.is_direct_lake_using_view = lambda self: False   # skips that one rule

current = labs.run_model_bpa(dataset=MODEL, workspace=MODEL_WORKSPACE, return_dataframe=True, export=False)
print("findings this scan:", len(current))


In [ ]:
os.makedirs(SCAN_DIR, exist_ok=True)

# The previous scan is the newest stamped file that exists BEFORE we write this one.
previous_files = sorted(glob.glob(os.path.join(SCAN_DIR, "bpa_results_*.csv")))
previous = pd.read_csv(previous_files[-1]) if previous_files else None

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
current = current.copy()
current["ScanUtc"] = stamp
current.to_csv(os.path.join(SCAN_DIR, f"bpa_results_{stamp}.csv"), index=False)
current.to_csv(os.path.join(SCAN_DIR, "bpa_results.csv"), index=False)   # always the latest, easy to point a report at
print("saved", f"bpa_results_{stamp}.csv", "| previous scan:", os.path.basename(previous_files[-1]) if previous_files else "none (first scan)")


In [ ]:
print("FINDINGS BY CATEGORY AND SEVERITY")
display(summarize(current))

changes = diff_scans(previous, current)
print(f"NEW since last scan: {len(changes['new'])}   FIXED: {len(changes['fixed'])}   UNCHANGED: {len(changes['unchanged'])}")
if previous is None:
    print("First scan, so everything counts as NEW. Run it again after the next deployment.")
print("\nNEW findings");   display(changes["new"])
print("FIXED findings");   display(changes["fixed"])


## Make it a habit

- Run this after each deployment or refresh (schedule the notebook).
- Alert on any **new** high-severity finding. Treat security findings (no row-level security,
  open permissions) as immediate.
- Watch the total as a trend, not a single number.
